# LangChain Complete TutorialA comprehensive notebook covering all key topics in the LangChain framework.## Table of Contents1. LangChain Overview2. Building Blocks (Chat Models, Prompts, Parsers)3. Chains (LLMChain, SequentialChain, RouterChain)4. Memory, Tools, and Agents5. Patterns and Best Practices6. Practical Examples

## 1. LangChain OverviewLangChain is an open-source framework for building applications with large language models (LLMs).### Key Features:- **Abstraction**: Simplified interfaces for LLM interactions- **Composability**: Chain components together- **Tool Ecosystem**: Built-in integrations for tools, vector databases- **Memory**: Built-in conversation memory- **Agents**: Autonomous agents with tool usage- **Production Features**: Debugging, monitoring, evaluation

In [ ]:
# Installation!pip install langchain langchain-openai langchain-community

### LangChain vs LangGraph| Aspect | LangChain | LangGraph ||--------|-----------|-----------|| Structure | Linear chains | Cyclic graphs || Use Case | Simple workflows | Complex, stateful agents || Control Flow | Sequential | Conditional, branching || State Management | Basic | Sophisticated || Best for | Prototypes | Production agents |

## 2. Building Blocks

### 2.1 Chat ModelsChat models are the core of LangChain applications. Supported providers include OpenAI, Anthropic, Azure OpenAI, Google Vertex AI, and HuggingFace.

In [ ]:
from langchain_openai import ChatOpenAIfrom langchain_anthropic import ChatAnthropic# OpenAI Chatllm = ChatOpenAI(model="gpt-4", temperature=0.7, max_tokens=1000)# Anthropic Chat# claude = ChatAnthropic(model="claude-3-opus")

### Key Parameters:- **temperature**: Controls randomness (0-2)- **max_tokens**: Maximum tokens in response- **model_name**: Which model to use- **streaming**: Enable streaming responses

### 2.2 Prompt TemplatesPrompt templates create reusable, parameterized prompts.

In [ ]:
from langchain.prompts import PromptTemplate, ChatPromptTemplate# String Prompt Templatestring_template = PromptTemplate.from_template("Tell me a {adjective} joke about {topic}")# Chat Prompt Templatechat_template = ChatPromptTemplate.from_messages([    ("system", "You are a {role} assistant."),    ("human", "{user_input}")])# Format promptsformatted_string = string_template.format(adjective="funny", topic="AI")formatted_chat = chat_template.format(role="helpful", user_input="What is AI?")

### 2.3 Output ParsersOutput parsers transform LLM responses into usable formats.

In [ ]:
from langchain.schema import StrOutputParserfrom langchain.output_parsers import JsonOutputParser, PydanticOutputParserfrom pydantic import BaseModel# String Parserstr_parser = StrOutputParser()# JSON Parserjson_parser = JsonOutputParser()# Pydantic Parserclass Recipe(BaseModel):    name: str    ingredients: list[str]    instructions: list[str]    prep_time_minutes: intpydantic_parser = PydanticOutputParser(pydantic_object=Recipe)

### 2.4 CachingCaching reduces costs and improves speed by storing LLM responses.

In [ ]:
from langchain.cache import InMemoryCachefrom langchain.globals import set_llm_cache# Enable in-memory cachingset_llm_cache(InMemoryCache())

### 2.5 StreamingStreaming enables real-time response display.

In [ ]:
# Streaming response (token-by-token)# for chunk in llm.stream("Tell me a story"):#     print(chunk.content, end="")

## 3. Chains (LCEL)Chains combine multiple components using the LangChain Expression Language (LCEL).

In [ ]:
# Simple LCEL Chainchain = string_template | llm | str_parserresult = chain.invoke({"adjective": "funny", "topic": "AI"})print(result)

### 3.1 LLMChainLLMChain is the basic chain combining prompt + LLM.

In [ ]:
from langchain.chains import LLMChain# LLMChain - Basic prompt + LLMllm_chain = LLMChain(llm=llm, prompt=string_template, output_key="joke")result = llm_chain.invoke({"adjective": "hilarious", "topic": "data science"})print(result["joke"])

### 3.2 SequentialChainSequentialChain runs multiple chains in sequence.

In [ ]:
from langchain.chains import SequentialChain, LLMChainfrom langchain.prompts import PromptTemplate# Chain 1: Generate factsfacts_prompt = PromptTemplate(    input_variables=["topic"],    template="List 3 interesting facts about {topic}")facts_chain = LLMChain(llm=llm, prompt=facts_prompt, output_key="facts")# Chain 2: Summarize factssummary_prompt = PromptTemplate(    input_variables=["facts"],    template="Summarize these facts into one sentence: {facts}")summary_chain = LLMChain(llm=llm, prompt=summary_prompt, output_key="summary")# Sequential Chainseq_chain = SequentialChain(    chains=[facts_chain, summary_chain],    input_variables=["topic"],    output_variables=["facts", "summary"],)result = seq_chain.invoke({"topic": "artificial intelligence"})print("Facts:", result["facts"])print("Summary:", result["summary"])

### 3.3 RouterChainRouterChain routes to different chains based on input.

In [ ]:
from langchain.chains import RouterChainfrom langchain.chains.llm import LLMChain# Define destination chainsphysics_prompt = PromptTemplate(template="Explain {topic} in physics")math_prompt = PromptTemplate(template="Explain {topic} in mathematics")physics_chain = LLMChain(llm=llm, prompt=physics_prompt)math_chain = LLMChain(llm=llm, prompt=math_prompt)# Create routerdefault_chain = LLMChain(llm=llm, prompt=PromptTemplate(template="{topic}"))router_chain = RouterChain(    default_chain=default_chain,    destination_chains={        "physics": physics_chain,        "math": math_chain    })

## 4. Memory TypesMemory allows chains to maintain state between interactions.

In [ ]:
from langchain.memory import (    ConversationBufferMemory,    ConversationBufferWindowMemory,    ConversationSummaryMemory,    ConversationTokenBufferMemory)# Buffer Memory - Store all messagesbuffer_memory = ConversationBufferMemory(    return_messages=True,    output_key="text",    input_key="input")# Window Memory - Last K messageswindow_memory = ConversationBufferWindowMemory(k=5, return_messages=True)# Summary Memory - Summarized historysummary_memory = ConversationSummaryMemory(llm=llm)

### Memory Types Summary:- **ConversationBufferMemory**: Store all messages- **ConversationBufferWindowMemory**: Last K messages only- **ConversationSummaryMemory**: Summarized conversation history- **ConversationTokenBufferMemory**: By token limit- **VectorStoreMemory**: Semantic retrieval from history

## 5. Tools and Agents

### 5.1 Creating ToolsTools extend agent capabilities with custom functions.

In [ ]:
from langchain.tools import tool@tooldef calculate(expression: str) -> str:    """Evaluate a math expression."""    try:        return str(eval(expression))    except Exception as e:        return f"Error: {str(e)}"@tooldef search(query: str) -> str:    """Search for information."""    return f"Results for: {query}"

### 5.2 Agent TypesAgents use an LLM to determine which actions to take.

In [ ]:
from langchain.agents import AgentType, initialize_agent# Zero-shot ReACT agentagent = initialize_agent(    tools=[calculate, search],    llm=llm,    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,    verbose=True)# Other agent types:# AgentType.CONVERSATIONAL_REACT_DESCRIPTION - With memory# AgentType.STRUCTURED_CHAT - Complex inputs# AgentType.SELF_ASK_WITH_SEARCH - Use search tool

### Agent Loop:1. Receive input2. Decide action3. Execute tool4. Observe result5. Repeat until done

## 6. Patterns and Best Practices

### 6.1 Error Handling

In [ ]:
# Error handling best practices:- Try/except: Wrap LLM calls- Retry logic: Use retry callbacks- Fallback chains: Alternate on failure- Timeout: Set max execution time- Circuit breaker: Prevent cascade failures

### 6.2 Production Optimization

In [ ]:
# Optimization strategies:- Caching: Cache LLM responses- Prompt optimization: Reduce tokens- Smaller models: Use cheaper models when possible- Batch processing: Group requests- Memory selection: Choose appropriate memory type

### 6.3 Runnable InterfaceThe standard interface for all LangChain components.

In [ ]:
# Runnable methods:- .invoke() - Synchronous single call- .batch() - Synchronous batch- .stream() - Streaming- .ainvoke() - Async single call- .abatch() - Async batch# Composition with pipe operator:chain = prompt | llm | output_parser

## 7. Practical Examples

### Example 1: Basic Chatbot with Memory

In [ ]:
from langchain.chains import ConversationChainfrom langchain.memory import ConversationBufferMemory# Create conversation chain with memoryconv = ConversationChain(    llm=llm,    memory=ConversationBufferMemory())# Interactresponse = conv.predict(input="Hi! I am John.")print(response)response = conv.predict(input="What is my name?")print(response)

### Example 2: Tool-using Agent

In [ ]:
# Create agent with calculator toolmath_agent = initialize_agent(    tools=[calculate],    llm=llm,    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION)# Use agentresult = math_agent.invoke("What is 5 + 3?")print(result)

### Example 3: RAG Chain (Question Answering)

In [ ]:
# RAG (Retrieval Augmented Generation)# from langchain.chains import RetrievalQA# from langchain.vectorstores import Chroma# from langchain.embeddings import OpenAIEmbeddings# Create vector store# vectorstore = Chroma.from_documents(documents, embedding=OpenAIEmbeddings())# retriever = vectorstore.as_retriever()# Create QA chain# qa_chain = RetrievalQA.from_chain_type(#     llm=llm,#     chain_type="stuff",#     retriever=retriever,#     return_source_documents=True# )

## 8. SummaryKey LangChain concepts covered:1. **Overview**: Framework, components, ecosystem2. **Building Blocks**: Models, prompts, parsers, caching, streaming3. **Chains**: LLMChain, SequentialChain, RouterChain, LCEL4. **Memory**: Buffer, Window, Summary, Token types5. **Tools & Agents**: Tool creation, agent types, agent loop6. **Best Practices**: Error handling, optimization, production